# Run Model

Single-country simulation runner with standard macro, benchmark, permanent-income, sector, and firm-credit diagnostics. Configure the inputs, run the workflow once, then inspect the manifest and diagnostic sections below.

## Companion notebooks

- `run_model_exploration.ipynb` — exploratory firm, household, reader, and ratio analysis
- `run_sensitivity.ipynb` — parameter sensitivity
- `run_mpc.ipynb` — household MPC experiment
- `run_irf.ipynb` — macro impulse responses
- `run_model_legacy_2026-08-18.ipynb` — preserved historical notebook; do not use for new work

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.monte_carlo import run_seeded_monte_carlo
from config import (
    BALANCE_SHEET_COLUMNS,
    FIGURE_SIZES,
    FISCAL_COLUMNS,
    LABOUR_COLUMNS,
    MACRO_COLUMNS,
    POLICY_COLUMNS,
    SCENARIO_PRESETS,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import (
    NotebookRunConfig,
    build_permanent_income_forecast_contribution_table,
    build_permanent_income_log_ratio_decomposition_df,
    plot_permanent_income_log_ratio_decomposition,
)
from src.visual_helpers import (
    firm_sector_groups_table,
    plot_agent_timeseries,
    plot_cumulative_insolvent_firms_by_sector,
    build_credit_demand_by_agent_df,
    build_credit_rationing_aggregates_df,
    plot_firm_credit_to_equity_and_capital,
    plot_mc,
    plot_output,
    plot_sector_tfp_investment_desired_mb_mc_ratio,
)

## Inputs

All execution switches are centralized here. Sensitivity, MPC, and IRF switches are in their dedicated notebooks.

In [2]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"
SCENARIO_NAME = "tfp_investment_adjustment_cost"

run_config = NotebookRunConfig(
    seed=40,
    t_max=50,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,  # unmodified France config: fixed alpha=0
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]

## Run simulation

In [3]:
# In the first run, state is none. After, run_notebook_workflow uses prepared from the previous state, preventing reloading it.
state = globals().get("state")

state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
    previous_state=state,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_scenario = state.df_scenario
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)

{'seed': 40, 'country': 'FRA', 't_max': 50, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                              

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 6 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+14.20%), industry 2 (+0.00%), industry 9 (+1.02%), industry 12 (+2.84%), industry 14 (+4.92%), industry 15 (+1.99%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()


Simulation complete. Output saved to: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                                  'hiring_speed': 0.75,
                                  'individuals_quitting': False,
                                  'individuals_quitting_temperature': 1.0,
                                  'optimised_hirin

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 6 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+14.20%), industry 2 (+0.00%), industry 9 (+1.02%), industry 12 (+2.84%), industry 14 (+4.92%), industry 15 (+1.99%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()


Simulation complete. Output saved to: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/FRA_benchmark.h5
Benchmark dataframe cached at: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/FRA_df_benchmark.pkl


In [4]:
{
    "country": COUNTRY,
    "seed": cfg.seed,
    "t_max": cfg.t_max,
    "scenario": state.scenario_name,
    "model_h5": str(simulation.model_h5_path),
    "benchmark": benchmark is not None,
    "manifest": str(state.manifest_path),
}


{'country': 'FRA',
 'seed': 40,
 't_max': 50,
 'scenario': 'tfp_investment_adjustment_cost',
 'model_h5': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5',
 'benchmark': True,
 'manifest': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/run_manifest_FRA_seed40_t50.json'}

## Macro, fiscal, policy, labour, and balance sheets

In [5]:
plot_output(
    df=df_scenario[list(MACRO_COLUMNS)],
    no_rows=5,
    no_cols=4,
    country_code=COUNTRY,
    line_color="#1f77b4",
    **FIGURE_SIZES["benchmark"],
)
plot_output(df=df_scenario[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(POLICY_COLUMNS)], no_rows=4, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(LABOUR_COLUMNS)], no_rows=2, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


In [6]:
from src.visual_helpers import plot_agent_timeseries  # noqa: E402


hh_balance_sheet_vars = [
    "liquid_financial_assets",
    "illiquid_financial_assets",
    "wealth_main_residence",
    "wealth_other_properties",  # other properties, not main residence
    "wealth_other_real_assets", # other real assets, not properties
    "wealth",  # gross wealth
    "mortgage_debt",
    "consumption_loan_debt",
    "debt",
    "net_wealth",  # wealth after liabilities
]

fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    hh_balance_sheet_vars,
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    # no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

## Benchmark comparison

In [7]:
if df_benchmark is None:
    print("Benchmark disabled in Inputs.")
else:
    plot_output(df=df_scenario[list(MACRO_COLUMNS)], df_compare=df_benchmark[list(MACRO_COLUMNS)], no_rows=10, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    # plot_output(df=df_scenario[list(FISCAL_COLUMNS)], df_compare=df_benchmark[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    # plot_output(df=df_scenario[list(POLICY_COLUMNS)], df_compare=df_benchmark[list(POLICY_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    # plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], df_compare=df_benchmark[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Fiscal diagnostics

In [8]:
def fiscal_diagnostic_table(df, items, periods=(1, 20, 50), unit=1e9, currency="€"):
    """Amount (bn) and % of GDP for each fiscal line item, at the given periods."""
    rows = {}
    for label, col in items.items():
        ratio_col = f"{col}_to_gdp"
        row = {}
        for t in periods:
            amount = df.loc[t, col] / unit
            pct = df.loc[t, ratio_col] * 100 if ratio_col in df.columns else float("nan")
            row[f"Period {t}"] = f"{currency}{amount:,.3f}bn ({pct:.2f}%)"
        rows[label] = row
    return pd.DataFrame(rows).T


REVENUE_ITEMS = {
    "Corporate income tax": "fiscal_revenue_corporate_income_taxes",
    "Household income tax": "fiscal_revenue_income_taxes",
    "Production tax": "fiscal_revenue_production_taxes",
    "VAT": "fiscal_revenue_vat",
    "Capital-formation taxes": "fiscal_revenue_capital_formation_taxes",
    "Export taxes": "fiscal_revenue_export_taxes",
    "Employee social insurance": "fiscal_revenue_employee_social_insurance",
    "Employer social insurance": "fiscal_revenue_employer_social_insurance",
    "Total revenue": "fiscal_revenue",
}

EXPENDITURE_ITEMS = {
    "Government consumption": "government_consumption",
    "Unemployment benefits": "unemployment_benefits",
    "Interest payments on debt": "interest_payments_on_debt",
    "Household social transfers": "household_social_transfers",
    "Public pension benefits": "public_pension_benefits",
    "Other social transfers": "other_social_transfers",
    "Necessity support": "necessity_support",
    "Total expenditure": "fiscal_expenditure",
}


from IPython.display import Markdown

fiscal_periods = (1, 20, 50)
revenue_table = fiscal_diagnostic_table(df_scenario, REVENUE_ITEMS, periods=fiscal_periods)
expenditure_table = fiscal_diagnostic_table(df_scenario, EXPENDITURE_ITEMS, periods=fiscal_periods)
print("Scenario tables")
display(Markdown("**Revenue**\n\n" + revenue_table.to_markdown()))
display(Markdown("**Expenditure**\n\n" + expenditure_table.to_markdown()))


REVENUES_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "fiscal_revenue_vat_to_gdp",
    "fiscal_revenue_production_taxes_to_gdp",
    "fiscal_revenue_capital_formation_taxes_to_gdp",
    "fiscal_revenue_corporate_income_taxes_to_gdp",
    "fiscal_revenue_income_taxes_to_gdp",
    "fiscal_revenue_rental_income_taxes_to_gdp",
    "fiscal_revenue_employee_social_insurance_to_gdp",
    "fiscal_revenue_employer_social_insurance_to_gdp",
    "fiscal_revenue_taxes_on_products_to_gdp",
    "fiscal_revenue_export_taxes_to_gdp",
    "fiscal_revenue_social_housing_rent_to_gdp",

    "fiscal_revenue_vat",
    "fiscal_revenue_production_taxes",
    "fiscal_revenue_capital_formation_taxes",
    "fiscal_revenue_corporate_income_taxes",
    "fiscal_revenue_income_taxes",
    "fiscal_revenue_rental_income_taxes",
    "fiscal_revenue_employee_social_insurance",
    "fiscal_revenue_employer_social_insurance",
    "fiscal_revenue_taxes_on_products",
    "fiscal_revenue_export_taxes",
    "fiscal_revenue_social_housing_rent",
    ]

EXPENDITURE_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "government_consumption_to_gdp",
    "unemployment_benefits_to_gdp",
    "interest_payments_on_debt_to_gdp",
    "household_social_transfers_to_gdp",
    "public_pension_benefits_to_gdp",
    "other_social_transfers_to_gdp",
    "necessity_support_to_gdp",

    "government_consumption",
    "unemployment_benefits",
    "interest_payments_on_debt",
    "household_social_transfers",
    "public_pension_benefits",
    "other_social_transfers",
    "necessity_support",
]

plot_output(df=df_scenario[list(REVENUES_COLS)], no_rows=7, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(EXPENDITURE_COLS)], no_rows=6, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


Scenario tables


**Revenue**

|                           | Period 1            | Period 20           | Period 50           |
|:--------------------------|:--------------------|:--------------------|:--------------------|
| Corporate income tax      | €29.749bn (5.33%)   | €31.323bn (4.78%)   | €39.651bn (5.47%)   |
| Household income tax      | €43.649bn (7.82%)   | €50.066bn (7.64%)   | €53.675bn (7.40%)   |
| Production tax            | €16.566bn (2.97%)   | €19.131bn (2.92%)   | €21.350bn (2.94%)   |
| VAT                       | €33.147bn (5.94%)   | €37.206bn (5.68%)   | €42.428bn (5.85%)   |
| Capital-formation taxes   | €4.561bn (0.82%)    | €22.811bn (3.48%)   | €24.736bn (3.41%)   |
| Export taxes              | €0.000bn (0.00%)    | €0.000bn (0.00%)    | €0.000bn (0.00%)    |
| Employee social insurance | €30.263bn (5.42%)   | €34.310bn (5.24%)   | €36.752bn (5.07%)   |
| Employer social insurance | €65.808bn (11.79%)  | €74.609bn (11.39%)  | €79.917bn (11.02%)  |
| Total revenue             | €223.744bn (40.09%) | €269.456bn (41.12%) | €298.510bn (41.17%) |

**Expenditure**

|                            | Period 1            | Period 20           | Period 50           |
|:---------------------------|:--------------------|:--------------------|:--------------------|
| Government consumption     | €148.587bn (26.63%) | €175.034bn (26.71%) | €198.162bn (27.33%) |
| Unemployment benefits      | €8.654bn (1.55%)    | €3.875bn (0.59%)    | €3.612bn (0.50%)    |
| Interest payments on debt  | €0.712bn (0.13%)    | €0.626bn (0.10%)    | €0.000bn (0.00%)    |
| Household social transfers | €169.427bn (30.36%) | €173.233bn (26.44%) | €183.461bn (25.30%) |
| Public pension benefits    | €133.939bn (24.00%) | €139.531bn (21.29%) | €146.814bn (20.25%) |
| Other social transfers     | €26.761bn (4.80%)   | €27.879bn (4.25%)   | €29.334bn (4.05%)   |
| Necessity support          | €0.072bn (0.01%)    | €1.948bn (0.30%)    | €3.702bn (0.51%)    |
| Total expenditure          | €318.725bn (57.11%) | €348.893bn (53.24%) | €381.623bn (52.63%) |

## Permanent-income decomposition

In [9]:
decomposition = build_permanent_income_log_ratio_decomposition_df(
    simulation,
    country_code=COUNTRY,
    reducer="mean",
    include_log_real_pc_income=True,
)
plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["ln_y_p_over_y", "common_log_ratio"],
    reducer="mean",
)

fig = plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["real_pc_income_idx"],
    reducer="mean",
    title='real_pc_income_idx',
    show=True,
)


contributions = build_permanent_income_forecast_contribution_table(
    simulation,
    country_code=COUNTRY,
    periods=list(range(9)),
    include_fixed=False,
)
contributions


,period,date,regressor,simulation_source,is_fixed,x_t,coefficient,contribution,point_forecast
0,0,2014Q1,time_trend,simulation_period_index,False,136.000000,0.004948,0.672896,0.022717
1,0,2014Q1,covid19,excluded_stage_3_dummy,False,0.000000,0.002059,0.000000,0.022717
2,0,2014Q1,log_real_pc_income,real_pc_income,False,4.605170,-1.017999,-4.688058,0.022717
3,0,2014Q1,d4_log_real_pc_income,real_pc_income,False,0.002550,0.068133,0.000174,0.022717
4,0,2014Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-0.757011,0.000638,-0.000483,0.022717
...,...,...,...,...,...,...,...,...,...
76,8,2016Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-1.554983,0.000638,-0.000992,0.026867
77,8,2016Q1,real_interest_rate_ma4_l5,policy_rate_and_cpi_fixed_basket,False,-0.487531,-0.002580,0.001258,0.026867
78,8,2016Q1,real_interest_rate_ma4_l9,policy_rate_and_cpi_fixed_basket,False,-0.996777,0.001678,-0.001673,0.026867
79,8,2016Q1,unemp_rate_ma4_l1,unemployment_rate,False,11.194805,-0.003852,-0.043122,0.026867


In [10]:
plot_agent_timeseries(
    model,
    "FRA",
    "households",
    variables = [
        'income',
        #  'non_property_income',
        'income_employee',
        # 'total_income_employee',
        'income_unemployment_benefits',
        'income_public_pension',
        'income_other_social_transfers',
        'income_social_transfers',
        # 'total_income_social_transfers',
        'income_rental',
        # 'total_income_rental',
        'income_dividend_distributions',
        # 'total_income_dividend_distributions'
        ],
    agg="mean",
    no_cols=3,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

In [11]:
# model.countries["FRA"].households.ts.get_keys()

## Validation

In [12]:
from src.validation import build_validation_availability_table, load_actual_policy_rate

# Availability check for the simulated-vs-actual comparison (autocorrelation and
# cross-correlation with real GDP), before any statistic is computed. See
# src/validation.py for series definitions, deflation, and rate-convention notes.
actual_policy_rate, policy_rate_available = load_actual_policy_rate(cfg, data)
validation_availability = build_validation_availability_table(data, df_scenario, COUNTRY, actual_policy_rate)
validation_availability

,simulated_column,simulated_available,actual_column,actual_available,both_available
series,,,,,
real_gdp,real_gdp,True,Real GDP (Value),True,True
real_consumption,household_consumption / ppi,True,Real Household Consumption (Value),True,True
real_investment,gfcf / ppi,True,Gross Fixed Capital Formation (Value) / PPI (V...,True,True
unemployment_rate,unemployment_rate,True,Unemployment Rate (Value),True,True
cpi,cpi_transaction,True,CPI (Value),True,True
central_bank_policy_rate,central_bank_policy_rate,True,"Policy Rate (BIS, quarterly, converted to per-...",True,True


In [13]:
from src.validation import build_validation_coverage_table, compute_validation_results

# User settings: how many lags to compute for each statistic.
MAX_ACF_LAGS = 12
MAX_CCF_LAGS = 8

validation_results, period_to_date = compute_validation_results(
    data, df_scenario, COUNTRY, actual_policy_rate, max_acf_lags=MAX_ACF_LAGS, max_ccf_lags=MAX_CCF_LAGS
)
build_validation_coverage_table(validation_results, n_total_periods=len(df_scenario.index))

real_gdp: actuals cover 38/51 model periods; statistics use the overlap only.
real_consumption: actuals cover 39/51 model periods; statistics use the overlap only.
real_investment: actuals cover 38/51 model periods; statistics use the overlap only.
unemployment_rate: actuals cover 29/51 model periods; statistics use the overlap only.
cpi: actuals cover 39/51 model periods; statistics use the overlap only.
central_bank_policy_rate: actuals cover 37/51 model periods; statistics use the overlap only.


,n_periods,n_total_periods
real_gdp,38,51
real_consumption,39,51
real_investment,38,51
unemployment_rate,29,51
cpi,39,51
central_bank_policy_rate,37,51


In [14]:
from src.validation import plot_validation_charts

# Charts: one row per validation variable, three panels each - level
# (simulated vs. actual on calendar years, dual y-axis), autocorrelation, and
# cross-correlation against real GDP.
plot_validation_charts(validation_results, period_to_date, COUNTRY).show()

## Sector and firm-credit diagnostics

### Sector structure and firm distress

Sector composition and insolvency counts by sector.


In [15]:
firm_sector_groups_table(model, COUNTRY)
plot_cumulative_insolvent_firms_by_sector(df_scenario)


### Firm credit demand, receipt, and leverage

Short-term and long-term requests are shown beside credit received. The final panel shows outstanding firm credit relative to equity and capital.


In [16]:
credit_panels = [
    ["total_target_short_term_credit", "total_received_short_term_credit"],
    ["total_target_long_term_credit", "total_received_long_term_credit"],
    "short_term_loan_debt",
    "long_term_loan_debt",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=credit_panels,
    agg="sum",
    no_cols=2,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

plot_firm_credit_to_equity_and_capital(model, COUNTRY, show=True, return_df=False)


### Investment
$$\text{total firm investment} = \text{capital-goods purchases} + \text{direct TFP spending} + \text{technical-investment spending},$$
$$\text{planned productivity investment} = \text{planned TFP investment} + \sum_{\text{inputs}}\text{planned technical investment}.$$

In [17]:
# Investment

investment_vars = [
    "total_capital_inputs_bought_costs",  # gross capital-goods purchases. (GFCF)
    # "net_capital_investment_above_replacement",  # gross capital purchases minus replacement/depreciation costs.
    # "planned_productivity_investment",  # total planned productivity investment.
    # "planned_tfp_investment",  # direct TFP component.
    # "planned_technical_investment",  # technical-coefficient component by input.
    "executed_tfp_investment",  # direct TFP investment actually executed.
    "executed_technical_investment",  # technical investment actually executed by input.
    ]

plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=investment_vars,
    agg="sum",
    no_cols=4,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

tfp_vars = [
    'tfp_investment_finance_scale',
    'planned_tfp_investment_expected_costs',
    'feasible_tfp_investment_expected_costs',
    'executed_productivity_investment',
    'net_capital_investment_above_replacement',
    'real_executed_productivity_investment',
    'planned_tfp_investment',
    'executed_tfp_investment',
    'direct_tfp_investment_cash_expense',
    'tfp_investment_effective_cost_rate',
    'tfp_investment_target_intensity',
    'tfp_investment_cap_intensity',
    'tfp_investment_desired_intensity',
    'tfp_investment_planned_intensity',
    'tfp_investment_desired_marginal_benefit',
    'tfp_investment_desired_marginal_cost',
    'tfp_investment_desired_mb_mc_ratio',
    'tfp_investment_desired_marginal_gap',
    'tfp_investment_planned_marginal_benefit',
    'tfp_investment_planned_marginal_cost',
    'tfp_investment_planned_mb_mc_ratio',
    'tfp_investment_planned_marginal_gap',
    'tfp_investment_cap_binding',
    'tfp_investment_cash_binding',
]

plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=tfp_vars,
    agg="sum",
    no_cols=4,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

plot_sector_tfp_investment_desired_mb_mc_ratio(model, COUNTRY)

## Firm balance-sheet diagnostics

The chart below shows period-end stock positions. The adjacent code defines transaction-account flow fields for follow-up plots.


In [18]:
# Balance-sheet variables (point-in-time stocks/positions)
balance_sheet_vars = [
    # Inventory and working capital
    'inventory',
    'inventory_nominal',
    'unsold_output',
    'intermediate_inputs_stock',
    'intermediate_inputs_stock_value',
    # 'intermediate_inputs_stock_industry',
    'capital_inputs_stock',
    'capital_inputs_stock_value',
    # 'capital_inputs_stock_industry',
    'expected_capital_inputs_stock_value',
    
    # Equity
    'equity',
    
    # Debt
    'short_term_loan_debt',
    'long_term_loan_debt',
    'debt',
    # 'total_debt',
    # 'total_credit_exposure',
    
    # Deposits and cash
    'deposits',
    # 'total_deposits',
    
    # Accumulated arrears/balances
    'firm_settlement_opening_interest_arrears',
    'firm_settlement_closing_interest_arrears',
    'firm_settlement_opening_principal_arrears',
    'firm_settlement_closing_principal_arrears',
    'operating_revolving_opening_balance',
    'operating_revolving_closing_balance',
]

# Flow variables (transactions over a period)
flow_vars = [
    # Production and output
    'production',
    'production_nominal',
    'output_by_employee_histogram',
    'total_sales',
    'expected_sales',
    'real_amount_sold',
    'real_amount_sold_to_FRA',
    'real_amount_sold_to_ROW',
    'nominal_amount_sold_in_lcu',
    'nominal_amount_sold_in_lcu_to_FRA',
    'nominal_amount_sold_in_lcu_to_ROW',
    
    # Profitability and income
    'profits',
    'expected_profits',
    'gross_operating_surplus_mixed_income',
    'real_wage_per_capita',
    
    # Costs and expenses
    'total_wage',
    'unit_costs',
    'labour_costs',
    'used_intermediate_inputs',
    'used_intermediate_inputs_costs',
    'total_intermediate_inputs_bought_costs',
    'used_capital_inputs',
    'used_capital_inputs_costs',
    'capital_depreciation_costs',
    'total_capital_inputs_bought_costs',
    
    # Taxes
    'taxes_paid_on_production',
    'corporate_taxes_paid',
    
    # Debt service (interest and principal)
    'interest_paid_on_deposits',
    'interest_paid_on_loans',
    'interest_paid',
    'firm_settlement_scheduled_interest_due',
    'firm_settlement_contractual_interest_due',
    'firm_settlement_payable_interest',
    'firm_settlement_unpaid_interest',
    'firm_settlement_capitalized_interest',
    'operating_revolving_interest_base',
    'operating_revolving_interest_paid',
    'debt_installments',
    'scheduled_debt_service',
    'total_debt_installments',
    'firm_settlement_scheduled_principal_due',
    'firm_settlement_contractual_principal_due',
    'firm_settlement_payable_principal',
    'firm_settlement_unpaid_principal',
    
    # Investment
    'planned_productivity_investment',
    'executed_productivity_investment',
    'net_capital_investment_above_replacement',
    'real_executed_productivity_investment',
    'planned_technical_investment',
    'executed_technical_investment',
    'technical_investment_by_input',
    'gross_fixed_capital_formation',
    'planned_tfp_investment',
    'executed_tfp_investment',
    'direct_tfp_investment_cash_expense',
    
    # Labour flows
    'labour_inputs',
    'desired_labour_inputs',
    'number_of_employees',
    
    # Demand and purchases
    'demand',
    'estimated_demand',
    'real_excess_demand',
    'accepted_goods_order',
    'goods_order',
    'real_amount_bought',
    'real_amount_bought_from_FRA',
    'real_amount_bought_from_ROW',
    'real_amount_bought_as_intermediate_inputs',
    'real_amount_bought_as_capital_goods',
    
    # Spending
    'nominal_amount_spent_in_usd',
    'nominal_amount_spent_in_usd_to_FRA',
    'nominal_amount_spent_in_usd_to_ROW',
    'nominal_amount_spent_in_lcu',
    'nominal_amount_spent_in_lcu_to_FRA',
    'nominal_amount_spent_in_lcu_to_ROW',
    
    # Inventory changes
    'total_inventory_change',
    
    # Dividends
    'dividend_fund_declared_distribution',
    
    # Credit received
    'received_short_term_credit',
    'total_received_short_term_credit',
    'received_debt_rollover_credit',
    'total_received_debt_rollover_credit',
    'received_overdraft_refinance_credit',
    'total_received_overdraft_refinance_credit',
    'received_operating_refinance_credit',
    'total_received_operating_refinance_credit',
    'received_ordinary_short_term_credit',
    'total_received_ordinary_short_term_credit',
    'received_long_term_credit',
    'total_received_long_term_credit',
    
    # Wage growth
    'wage_offer_growth',
    'wage_offer_growth_historic_wage',
    'wage_offer_growth_tfp',
    'wage_offer_growth_productivity',
    'wage_offer_growth_tightness',
    'wage_offer_growth_residual',
]

plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=balance_sheet_vars,
    agg="sum",
    no_cols=3,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

## Bank diagnostics

Borrower demand is requested new credit; realized supply is credit granted and recorded as received in that period. A **bank supply cap** is a separate CAR-based maximum lending capacity allocated across loan types. It is not actual lending. The firm short-term cap is split using ordinary short-term demand and excludes emergency overdraft refinance.


### Credit demand and supply by borrower and product

Each panel compares demand with the corresponding bank supply cap. Amounts are summed across agents and banks.


In [19]:
from IPython.display import display

# Aggregate borrower-side requests and granted credit separately from bank capacity.
credit_diagnostics = build_credit_rationing_aggregates_df(model, COUNTRY)
firm_credit_by_t = build_credit_demand_by_agent_df(
    model, COUNTRY, agent_kind="firms", include_received=True
).groupby(level="t").sum(min_count=1)
household_credit_by_t = build_credit_demand_by_agent_df(
    model, COUNTRY, agent_kind="households", include_received=True
).groupby(level="t").sum(min_count=1)

credit_products = [
    {
        "label": "Firms: short-term",
        "demand": firm_credit_by_t["target_short_term_credit"],
        "received": firm_credit_by_t["received_short_term_credit"],
        "cap": credit_diagnostics["bank_credit_supply_cap_firms_short_term"],
        "cap_shortfall": credit_diagnostics["bank_credit_supply_gap_firms_short_term"],
        "cap_rate": credit_diagnostics["bank_credit_rationing_rate_firms_short_term"],
    },
    {
        "label": "Firms: long-term",
        "demand": firm_credit_by_t["target_long_term_credit"],
        "received": firm_credit_by_t["received_long_term_credit"],
        "cap": credit_diagnostics["bank_credit_supply_cap_firms_long_term"],
        "cap_shortfall": credit_diagnostics["bank_credit_supply_gap_firms_long_term"],
        "cap_rate": credit_diagnostics["bank_credit_rationing_rate_firms_long_term"],
    },
    {
        "label": "Households: consumption loans",
        "demand": household_credit_by_t["target_consumption_loans"],
        "received": household_credit_by_t["received_consumption_loans"],
        "cap": credit_diagnostics["bank_credit_supply_cap_households_consumption"],
        "cap_shortfall": credit_diagnostics["bank_credit_supply_gap_households_consumption"],
        "cap_rate": credit_diagnostics["bank_credit_rationing_rate_households_consumption"],
    },
    {
        "label": "Households: mortgages",
        "demand": household_credit_by_t["target_mortgage"],
        "received": household_credit_by_t["received_mortgages"],
        "cap": credit_diagnostics["bank_credit_supply_cap_mortgages"],
        "cap_shortfall": credit_diagnostics["bank_credit_supply_gap_mortgages"],
        "cap_rate": credit_diagnostics["bank_credit_rationing_rate_mortgages"],
    },
]

peak_rows = []
for product in credit_products:
    demand = pd.to_numeric(product["demand"], errors="coerce")
    valid_demand = demand.dropna()
    if valid_demand.empty:
        continue

    peak_t = valid_demand.idxmax()
    peak_demand = float(demand.loc[peak_t])
    received = pd.to_numeric(product["received"], errors="coerce").reindex(demand.index).loc[peak_t]
    bank_cap = pd.to_numeric(product["cap"], errors="coerce").reindex(demand.index).loc[peak_t]
    cap_shortfall = pd.to_numeric(product["cap_shortfall"], errors="coerce").reindex(demand.index).loc[peak_t]
    cap_rate = pd.to_numeric(product["cap_rate"], errors="coerce").reindex(demand.index).loc[peak_t]
    peak_rows.append(
        {
            "Product": product["label"],
            "Peak t": peak_t,
            "Demand": peak_demand,
            "Credit granted/received": received,
            "Unmet demand": max(peak_demand - received, 0.0) if pd.notna(received) else np.nan,
            "Bank cap": bank_cap,
            "Demand above cap": cap_shortfall,
            "Cap shortfall (%)": 100 * cap_rate if pd.notna(cap_rate) else np.nan,
        }
    )

bank_credit_peak_summary = pd.DataFrame(peak_rows).set_index("Product")
print("Peak-demand period by product. Credit granted/received is realized lending; bank cap is a separate ceiling.")
display(bank_credit_peak_summary.style.format(precision=3, thousands=","))

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[product["label"] for product in credit_products],
    shared_xaxes=True,
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)
credit_traces = [
    ("demand", "Borrower demand", "#2563eb"),
    ("received", "Credit granted/received", "#16a34a"),
    ("cap", "Bank cap (CAR capacity)", "#f59e0b"),
]
for idx, product in enumerate(credit_products):
    row = idx // 2 + 1
    col = idx % 2 + 1
    for key, label, color in credit_traces:
        fig.add_trace(
            go.Scatter(
                x=product[key].index,
                y=product[key],
                mode="lines",
                name=label,
                legendgroup=key,
                showlegend=(idx == 0),
                line={"color": color, "width": 2},
            ),
            row=row,
            col=col,
        )
    fig.update_yaxes(title_text="Credit per period", rangemode="tozero", row=row, col=col)
    fig.update_xaxes(title_text="t", row=row, col=col)

fig.update_layout(
    title_text=f"{COUNTRY}: credit demand, realized lending, and bank capacity",
    template="plotly_white",
    hovermode="x unified",
    height=680,
    width=1180,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
    margin={"t": 125, "b": 50},
)
fig.show()


Peak-demand period by product. Credit granted/received is realized lending; bank cap is a separate ceiling.


,Peak t,Demand,Credit granted/received,Unmet demand,Bank cap,Demand above cap,Cap shortfall (%)
Product,,,,,,,
Firms: short-term,3,"543,218,357.881","543,218,357.881",0.000,0.000,"543,218,357.881",100.000
Firms: long-term,2,"78,927,626,778.450","69,518,725,848.298","9,408,900,930.152","4,465,147,213,003.158",0.000,0.000
Households: consumption loans,49,"53,258,270,338.638","3,636,910,692.946","49,621,359,645.692","430,788,644,787.963",0.000,0.000
Households: mortgages,41,"58,350,448,897.847","13,366,252,559.894","44,984,196,337.953","1,935,843,360,165.423",0.000,0.000


### Realized rationing and cap shortfall

Realized rationing compares demand with credit granted/received. Cap shortfall compares demand with the bank capacity ceiling; it shows whether that ceiling could bind. The two can differ because borrower eligibility and other underwriting limits also affect lending.


In [20]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[product["label"] for product in credit_products],
    shared_xaxes=True,
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)
for idx, product in enumerate(credit_products):
    row = idx // 2 + 1
    col = idx % 2 + 1
    demand = pd.to_numeric(product["demand"], errors="coerce")
    received = pd.to_numeric(product["received"], errors="coerce").reindex(demand.index)
    realized_shortfall = (demand - received).clip(lower=0.0)
    realized_rate = pd.Series(np.nan, index=demand.index, dtype=float)
    no_demand = demand <= 0
    realized_rate.loc[no_demand] = 0.0
    observed = (demand > 0) & received.notna()
    realized_rate.loc[observed] = (
        realized_shortfall.loc[observed] / demand.loc[observed] * 100
    )
    cap_rate = pd.to_numeric(product["cap_rate"], errors="coerce").reindex(demand.index) * 100

    for values, name, color in [
        (realized_rate, "Unmet after realized lending", "#16a34a"),
        (cap_rate, "Shortfall implied by bank cap", "#f59e0b"),
    ]:
        fig.add_trace(
            go.Scatter(
                x=values.index,
                y=values,
                mode="lines",
                name=name,
                legendgroup=name,
                showlegend=(idx == 0),
                line={"color": color, "width": 2},
            ),
            row=row,
            col=col,
        )
    fig.update_yaxes(title_text="Unmet share (%)", range=[0, 100], row=row, col=col)
    fig.update_xaxes(title_text="t", row=row, col=col)

fig.update_layout(
    title_text=f"{COUNTRY}: realized credit shortfall vs bank-cap constraint",
    template="plotly_white",
    hovermode="x unified",
    height=680,
    width=1180,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
    margin={"t": 125, "b": 50},
)
fig.show()


### Bank balance sheet

Loan portfolios are separated by borrower/product; deposits, total liabilities, and equity show the funding and capital side. These are bank-level positions summed across banks.


In [21]:
bank_loan_balance_panels = [
    ["short_term_loans_to_firms", "revolving_operating_facility_exposure"],
    "long_term_loans_to_firms",
    "consumption_loans_to_households",
    "mortgages_to_households",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "banks",
    variables=bank_loan_balance_panels,
    panel_titles=[
        "Firm short-term and revolving credit",
        "Firm long-term loans",
        "Household consumption loans",
        "Household mortgages",
    ],
    no_cols=2,
    show_legend=True,
    **FIGURE_SIZES["dense"],
)

plot_agent_timeseries(
    model,
    COUNTRY,
    "banks",
    variables=["deposits", "liability", "equity"],
    agg="sum",
    no_cols=3,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)


## Household Diagnostics

Read the saved scenario and benchmark; these cells **do not rerun the model**. Solid lines are the scenario, dashed lines the benchmark. Labels identify the active scenario; consult the manifest for the benchmark definition.

- Money: nominal local currency, billions; flows per model period, stocks at period end.
- Shares and quantiles: unweighted simulated households. Rates and portfolio shares are averaged, never summed.
- Total consumption includes actual and imputed housing services; cash consumption excludes imputed rent. Goods orders/purchases are compared separately, before VAT.
- Missing datasets remain unavailable; coverage is listed below. Initial placeholders for decision diagnostics are hidden.


In [22]:
from IPython.display import display
from src.household_diagnostics import (
    build_household_diagnostics,
    household_diagnostic_summary,
    plot_household_diagnostics,
)

household_runs = {}
household_coverage = {}
scenario_label = f"Scenario ({state.scenario_name})"
household_runs[scenario_label], household_coverage[scenario_label] = build_household_diagnostics(
    simulation, country_code=COUNTRY, index=df_scenario.index,
)
if benchmark is not None:
    household_runs["Benchmark"], household_coverage["Benchmark"] = build_household_diagnostics(
        benchmark, country_code=COUNTRY, index=df_benchmark.index,
    )

display(household_diagnostic_summary(household_runs).round(4))
coverage = pd.concat(household_coverage, names=["run", "dataset"])
display(coverage.loc[(coverage["status"] != "available") | (coverage["finite_pct"] < 100)])


consumption_including_housing  \
run                                       observation                                   
Scenario (tfp_investment_adjustment_cost) initial                            309.8223   
                                          first period                       301.2113   
                                          final                              381.4112   
Benchmark                                 initial                            309.8223   
                                          first period                       301.2113   
                                          final                              382.9496   

                                                        expected_income  \
run                                       observation                     
Scenario (tfp_investment_adjustment_cost) initial              329.2137   
                                          first period         324.9467   
                                          final                372.7172   
Benchmark                                 initial              329.2137   
                                          first period         324.9467   
                                          final                375.2546   

                                                          income  \
run                                       observation              
Scenario (tfp_investment_adjustment_cost) initial       329.2137   
                                          first period  324.9959   
                                          final         374.7660   
Benchmark                                 initial       329.2137   
                                          first period  324.9959   
                                          final         376.0181   

                                                        income_forecast_shortfall  \
run                                       observation                               
Scenario (tfp_investment_adjustment_cost) initial                             NaN   
                                          first period                     0.0232   
                                          final                            1.6527   
Benchmark                                 initial                             NaN   
                                          first period                     0.0232   
                                          final                            0.0882   

                                                        liquid_financial_assets  \
run                                       observation                             
Scenario (tfp_investment_adjustment_cost) initial                      529.6249   
                                          first period                 490.9299   
                                          final                        562.6193   
Benchmark                                 initial                      529.6249   
                                          first period                 490.9299   
                                          final                        528.1672   

                                                        overdrafts  \
run                                       observation                
Scenario (tfp_investment_adjustment_cost) initial              0.0   
                                          first period         0.0   
                                          final                0.0   
Benchmark                                 initial              0.0   
                                          first period         0.0   
                                          final                0.0   

                                                        overdraft_share  \
run                                       observation                     
Scenario (tfp_investment_adjustment_cost) initial                   0.0   
                                          first period           

status  \
run                                       dataset                                          
Scenario (tfp_investment_adjustment_cost) cash_saving_before_financing         available   
                                          interest_paid                        available   
                                          interest_paid_on_deposits            available   
                                          interest_paid_on_loans               available   
                                          portfolio_liquid_return_rate         available   
                                          received_consumption_loans           available   
                                          received_mortgages                   available   
                                          target_consumption_calibrated_total  available   
                                          target_consumption_loans             available   
Benchmark                                 cash_saving_before_financing         available   
                                          interest_paid                        available   
                                          interest_paid_on_deposits            available   
                                          interest_paid_on_loans               available   
                                          portfolio_liquid_return_rate         available   
                                          received_consumption_loans           available   
                                          received_mortgages                   available   
                                          target_consumption_calibrated_total  available   
                                          target_consumption_loans             available   

                                                                               finite_pct  
run                                       dataset                                          
Scenario (tfp_investment_adjustment_cost) cash_saving_before_financing          98.039216  
                                          interest_paid                         98.039216  
                                          interest_paid_on_deposits             98.039216  
                                          interest_paid_on_loans                98.039216  
                                          portfolio_liquid_return_rate           1.960784  
                                          received_consumption_loans            98.039216  
                                          received_mortgages                    98.039216  
                                          target_consumption_calibrated_total   98.039216  
                                          target_consumption_loans              98.039216  
Benchmark                                 cash_saving_before_financing          98.039216  
                                          interest_paid                         98.039216  
                                          interest_paid_on_deposits             98.039216  
                                          interest_paid_on_loans                98.039216  
                                          portfolio_liquid_return_rate           1.960784  
                                          received_consumption_loans            98.039216  
                                          received_mortgages                    98.039216  
                                          target_consumption_calibrated_total   98.039216  
                                          target_consumption_loans              98.039216

### Consumption and income

Inspect the initial consumption jump, the long-run adjustment contribution, and the expected-versus-realised cash-income gap. Emergency support is separated from the forecast error.


In [23]:
household_fig = plot_household_diagnostics(household_runs, section='Consumption and income')
household_fig.show()


### Credit and payment constraints

Compare formal credit requests and grants, service payments, financing repairs and constraint incidence. Realised service/income is an outcome diagnostic, not the underwriting DSTI test: it includes existing and early principal payments and excludes overdraft interest.


In [24]:
household_fig = plot_household_diagnostics(household_runs, section='Credit and payment constraints')
household_fig.show()


### Balance sheets and overdrafts

Distinguish positive deposits, negative deposits and explicit debt. Total exposure adds overdrafts to mortgage and consumer debt. Compare overdrafts before and after portfolio settlement to locate their origin.


In [25]:
household_fig = plot_household_diagnostics(household_runs, section='Balance sheets and overdrafts')
household_fig.show()


### Cash ledger and portfolio

Compare saving before finance, new loans and liquid-asset changes. Expenditure includes investment as well as goods consumption; cash consumption includes rent. The cash identity residual checks bookkeeping, not economic feasibility. Portfolio flows shown here are committed transactions, not counterfactual targets.


In [26]:
household_fig = plot_household_diagnostics(household_runs, section='Cash ledger and portfolio')
household_fig.show()



## Optional Monte Carlo

In [27]:
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(run_config.seed)
    mc_seeds = rng.choice(np.arange(1000), size=50, replace=False).tolist()
    mc = run_seeded_monte_carlo(
        datawrapper=data,
        country_configurations=state.country_configurations,
        country_code=COUNTRY,
        seeds=mc_seeds,
        t_max=cfg.t_max,
        n_jobs=-4,
        backend="loky",
        batch_size=1,
    )
    plot_mc(mc=mc, cols=list(MACRO_COLUMNS), no_cols=4, country_code=COUNTRY, base_width=500)
else:
    print("Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.")


Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.


In [28]:
# Verify consistency ALM PLM

sigma_2_v_min = 0.0001
sigma_2_v_max = 0.0013
sigma_2_v_max = 0.0005
rho = 0.951
sigma_2_eta_a = 0.029
sigma_2_beta_a = 0.00038

sigma_2_eta_q = (1-rho**2) * sigma_2_eta_a / (1- rho**8) 
# print(sigma_2_eta_q**(1/2))
# print(sigma_2_beta_a / 16)
print(sigma_2_eta_q)

sigma_2_eta_implied_min = (sigma_2_v_min - (1 - rho)**2 * sigma_2_beta_a) / (rho**4 - 1)**2
print(sigma_2_eta_implied_min)

sigma_2_eta_implied_max = (sigma_2_v_max - (1 - rho)**2 * sigma_2_beta_a) / (rho**4 - 1)**2
print(sigma_2_eta_implied_max)



0.008376445599508184
0.002989481681828435
0.015057514729058492
